In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Example Data Generation: Store Performance Model

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/analyze_site_performance/places_insights_analyze_site_performance_data_generation.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fplaces_insights%2Fnotebooks%2Fanalyze_site_performance%2Fplaces_insights_analyze_site_performance_data_generation.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/analyze_site_performance/places_insights_analyze_site_performance_data_generation.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/places_insights/notebooks/analyze_site_performance/places_insights_analyze_site_performance_data_generation.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

# Example Data Generation: Store Performance Model

### **Overview**
This notebook serves as the data generation engine for the **Analyze Site Performance with Google Places Insights and BigQuery ML** notebook.

Instead of relying on pre-canned data, we demonstrate how to create a realistic training dataset from scratch. We generate randomized store locations in London and perform a geospatial join with the Places Insights dataset. This allows us to calculate synthetic performance scores based on the density of amenities (gyms, restaurants, transit, etc.) surrounding each specific point.

**Key Features of this Notebook:**
*   **Real-time Scoring:** Dynamically calculate store performance based on proximity to real-world places.
*   **Visual Verification:** Interactively explore the generated data on a **Google Map** to sanity-check the spatial distribution and performance hotspots.
*   **Data Export:** Download the final dataset as a CSV file to be used in the main analysis notebook.

### **The Methodology**
To simulate realistic business metrics, we model the **Store Performance** ($Y$) as a linear function of the surrounding environment amenities, calculated using a **Multiple Linear Regression** approach.

The performance score is determined by the count of specific amenities (Predictors) within a **500m radius** of each store, plus a noise term.

The mathematical model is defined as:

$$
Y = \beta_0 + \beta_1 X_{\text{gym}} + \beta_2 X_{\text{restaurant}} + \beta_3 X_{\text{school}} + \beta_4 X_{\text{transit}} + \beta_5 X_{\text{clothing}} + \epsilon
$$

**Where:**

* $Y$: **Store Performance** (Response Variable), clipped to range $[0, 100]$.
* $\beta_0$: **Intercept**, set to a base value of **20**.
* $X_i$: **Predictors**, representing the count of operational places within 500m.
* $\beta_i$: **Coefficients** (weights) assigned to each predictor:
    * $\beta_1 = 0.2$ (Gyms)
    * $\beta_2 = 0.4$ (Restaurants)
    * $\beta_3 = 0.1$ (Schools)
    * $\beta_4 = 0.1$ (Transit Stations)
    * $\beta_5 = 0.2$ (Clothing Stores)
* $\epsilon$: **Error Term** (Noise), added to introduce variance.

### **Prerequisites & Setup**

*   **Google Cloud Project:** You need a project with BigQuery enabled.
*   **Places Insights Access:** Your project must be subscribed to the [GB Places Insights dataset](https://developers.google.com/maps/documentation/placesinsights/cloud-setup) in BigQuery sharing.
*   **Google Maps Platform [API Key](https://developers.google.com/maps/get-started):** Required to render the final interactive map visualization. Enable the [**Maps JavaScript API**](https://developers.google.com/maps/documentation/javascript/get-api-key?setupProd=enable#enable-the-api) and [**Maps Tiles API**](https://developers.google.com/maps/documentation/tile/get-api-key?setupProd=enable#enable-the-api) on this key.
*   **Colab Secrets:** Please add the following to the **Secrets** tab (Key icon on the left):
    *   `GCP_PROJECT_ID`: Your Google Cloud Project ID.
    *   `GMP_API_KEY`: The Google Maps API Key you configured in the previous step.

In [ ]:
# @title 1. Setup & Authentication
# @markdown Authenticate to Google Cloud, retrieve secrets, and initialize the BigQuery client.
import random
import requests
import pandas as pd
import geopandas as gpd
import folium
from folium import Element
from google.cloud import bigquery
from shapely import wkt
import google.auth
import getpass

try:
    # --- ATTEMPT 1: Standard Consumer Colab ---
    from google.colab import auth, userdata

    # 1. Retrieve Secrets
    GCP_PROJECT_ID = userdata.get('GCP_PROJECT_ID').strip()
    print(f"✅ Secrets retrieved for project: {GCP_PROJECT_ID}")

    GMP_API_KEY = userdata.get('GMP_API_KEY').strip()
    print("✅ GMP API Key retrieved.")

    # 2. Authenticate User
    print("🔄 Authenticating user...")
    auth.authenticate_user(project_id=GCP_PROJECT_ID)
    print("✅ User Authenticated.")

    # 3. Initialize BigQuery Client
    client = bigquery.Client(project=GCP_PROJECT_ID)

except (ImportError, Exception) as e:
    # --- ATTEMPT 2: Colab Enterprise / Local Jupyter ---
    print("ℹ️ Standard Colab setup skipped. Falling back to Enterprise/Local Auth...")

    # 1. Retrieve Environment Authentication & Project ID automatically
    credentials, GCP_PROJECT_ID = google.auth.default()
    print(f"✅ Authenticated via default credentials to Project: {GCP_PROJECT_ID}")

    # 2. Securely prompt for the API key
    print("🔑 Please paste your Google Maps Platform API Key and press Enter:")
    GMP_API_KEY = getpass.getpass().strip()
    print("✅ GMP API Key securely captured.")

    # 3. Initialize BigQuery Client with enterprise credentials
    client = bigquery.Client(credentials=credentials, project=GCP_PROJECT_ID)

print("✅ BigQuery Client Initialized.")

In [ ]:
# @title 2. Generate Synthetic Data & Calculate Scores
# @markdown Note: This cell takes ~2 minutes to execute.
from shapely import wkt

print("Generating synthetic locations in London...")

# 1. Generate Random Locations & Noise
LAT_MIN, LAT_MAX = 51.30, 51.70
LNG_MIN, LNG_MAX = -0.50, 0.30

sql_structs = []

for i in range(1, 501):
    lng = random.uniform(LNG_MIN, LNG_MAX)
    lat = random.uniform(LAT_MIN, LAT_MAX)
    noise = random.gauss(0, 2)

    # STRUCT construction
    sql_structs.append(
        f"STRUCT('STORE_{i:03d}' as store_id, ST_GEOGPOINT({lng:.4f}, {lat:.4f}) as location, {noise:.4f} as noise)"
    )

generated_data_sql = ",\n".join(sql_structs)

# 2. Construct Query
# We convert location to Text (ST_ASTEXT) to allow Grouping
query = f"""
WITH t AS (
    SELECT * FROM UNNEST([
        {generated_data_sql}
    ])
)
SELECT WITH AGGREGATION_THRESHOLD
    t.store_id,

    ST_ASTEXT(t.location) as location_wkt,

    -- Linear Model
    GREATEST(0, LEAST(100,
      20 +
      (0.2 * COUNTIF('gym' IN UNNEST(p.types))) +
      (0.4 * COUNTIF('restaurant' IN UNNEST(p.types))) +
      (0.1 * COUNTIF('school' IN UNNEST(p.types))) +
      (0.1 * COUNTIF('transit_station' IN UNNEST(p.types))) +
      (0.2 * COUNTIF('clothing_store' IN UNNEST(p.types))) +
      AVG(t.noise)
    )) AS store_performance
FROM
    t
LEFT JOIN
    `places_insights___gb.places` AS p
    ON ST_DWITHIN(t.location, p.point, 500)
    AND p.business_status = 'OPERATIONAL'
GROUP BY
    t.store_id, location_wkt
"""

print("Executing BigQuery GIS join...")

# 3. Execute to standard DataFrame (not GeoDataFrame yet)
df = client.query(query).to_dataframe()

# 4. Convert WKT String back to Geometry object
df['geometry'] = df['location_wkt'].apply(wkt.loads)

# 5. Convert to GeoDataFrame
df_stores = gpd.GeoDataFrame(df, geometry='geometry')

# 6. Cleanup: Drop redundant text column and rename geometry to 'location'
# This matches the schema expected by BigQuery in the subsequent notebook.
df_stores = df_stores.drop(columns=['location_wkt'])
df_stores = df_stores.rename_geometry('location')

print(f"✅ Generated and scored {len(df_stores)} stores.")
display(df_stores.head())

In [ ]:
# @title 3. Maps backend Initialization: Session, Copyright & Assets
# @markdown This cell manages the API handshake. It performs the following steps:
# @markdown 1.  **Session Creation:** Authenticates and requests a "Roadmap" session for the target region.
# @markdown 2.  **Attribution Fetching:** Queries the API for the specific copyright text required for the configured viewport.
# @markdown 3.  **Asset Preparation:** Generates the compliant HTML for the Google Maps logo overlay.

# --- 1. Create Google Maps Session ---
print("🗺️ Initializing Google Maps Session...")
session_url = f"https://tile.googleapis.com/v1/createSession?key={GMP_API_KEY}"
headers = {"Content-Type": "application/json"}
payload = {
    "mapType": "roadmap",
    "language": "en-GB",
    "region": "GB"
}

try:
    response = requests.post(session_url, json=payload, headers=headers)
    response.raise_for_status()
    session_token = response.json().get("session")
    print(f"✅ Session Token acquired.")
except Exception as e:
    raise RuntimeError(f"Failed to initialize Google Maps session: {e}")

# --- 2. Fetch Dynamic Attribution for London ---
# Center of our synthetic data area
LAT, LNG = 51.5074, -0.1278
ZOOM_LEVEL = 11
delta = 0.2

viewport_url = (
    f"https://tile.googleapis.com/tile/v1/viewport?key={GMP_API_KEY}"
    f"&session={session_token}"
    f"&zoom={ZOOM_LEVEL}"
    f"&north={LAT + delta}&south={LAT - delta}"
    f"&west={LNG - delta}&east={LNG + delta}"
)

try:
    vp_response = requests.get(viewport_url)
    vp_response.raise_for_status()
    google_attribution = vp_response.json().get('copyright', 'Map data © Google')
    print("✅ Attribution fetched.")
except Exception as e:
    print(f"⚠️ Warning: Could not fetch attribution ({e}). Defaulting.")
    google_attribution = "Map data © Google"

# --- 3. Construct Logo HTML ---
logo_url = "https://maps.gstatic.com/mapfiles/api-3/images/google_white3.png"
logo_html = f"""
    <div style="
        position: fixed;
        bottom: 35px;
        left: 10px;
        z-index: 9999;
        font-size: 0;
        pointer-events: none;
    ">
        <img src="{logo_url}" alt="Google Maps" style="height: 18px;">
    </div>
"""
print("✅ Logo HTML prepared.")

In [ ]:
# @title 4. Display Map with Data Overlay
# @markdown Renders the interactive map with the following components:
# @markdown *   **Google Maps Vector Tiles:** As the base layer.
# @markdown *   **Store Data Overlay:** Points colored by their calculated `store_performance` score (Purple=Low, Yellow=High).
# @markdown *   **Custom Controls:** A performance score legend and the required Google logo.

# --- 1. Construct Tiles URL ---
tiles_url = f"https://tile.googleapis.com/v1/2dtiles/{{z}}/{{x}}/{{y}}?session={session_token}&key={GMP_API_KEY}"

# --- 2. Initialize Map ---
m = folium.Map(
    location=[LAT, LNG],
    zoom_start=ZOOM_LEVEL,
    tiles=tiles_url,
    attr=google_attribution,
    name="Google Maps",
    control_scale=True,
    prefer_canvas=True
)

# --- 3. Add Google Logo (Bottom Left) ---
m.get_root().html.add_child(Element(logo_html))

# --- 4. Add Custom Legend/Key (Bottom Right) ---
# We use a CSS linear-gradient that matches the 'viridis' colormap used below
legend_html = """
<div style="
    position: fixed;
    bottom: 35px;
    right: 10px;
    width: 160px;
    height: 80px;
    z-index: 9999;
    background-color: rgba(255, 255, 255, 0.9);
    border-radius: 4px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.3);
    padding: 10px;
    font-family: Roboto, Arial, sans-serif;
    font-size: 12px;
">
    <strong style="font-size:13px;">Performance Score</strong>
    <div style="
        width: 100%;
        height: 12px;
        /* Viridis Color Scale: Purple -> Teal -> Green -> Yellow */
        background: linear-gradient(to right, #440154, #3b528b, #21918c, #5ec962, #fde725);
        margin-top: 8px;
        margin-bottom: 4px;
    "></div>
    <div style="display: flex; justify-content: space-between; color: #555;">
        <span>Low (~20)</span>
        <span>High (~80)</span>
    </div>
</div>
"""
m.get_root().html.add_child(Element(legend_html))

# --- 5. Overlay Store Data ---
df_stores.explore(
    m=m, # Add to our Google Map instance
    column='store_performance',
    vmin=20,
    vmax=80,
    scheme='NaturalBreaks',
    marker_kwds={"radius": 8, "fillOpacity": 0.8},
    cmap='viridis',
    tooltip=['store_id', 'store_performance'],
    name="Store Performance"
)

# Add layer control to toggle data on/off
folium.LayerControl().add_to(m)

display(m)

In [ ]:
# @title 5. Export Data to CSV
from google.colab import files

# 1. Save DataFrame to CSV in the Colab virtual machine
filename = 'store_performance_london.csv'
df_stores.to_csv(filename, index=False)
print(f"✅ Saved {filename} to runtime.")

# 2. Trigger download to your local machine
files.download(filename)